# Train the detector on Kaggle

This notebook is deliberately thin. A run is defined by `configs/train.yaml` and not by the
order these cells happen to be executed in — the notebook attaches the data, installs the
package and calls one command. If you want to change the schedule, the subset or the anchors,
change the config in the repository and commit it, so that the run stays reproducible from a
file rather than from a browser tab.

**Before running:**

1. Add the LS-SSDD-v1.0 dataset as an input. `data.root` in the config points at
   `/kaggle/input/ls-ssdd-v10/LS-SSDD-v1.0-OPEN`; if the attachment lands somewhere else,
   the first cell below prints where it actually is and the config is the thing to correct.
2. Turn the **Internet** switch on, in the session settings. It is needed once, to fetch the
   COCO backbone weights. Without it, set `model.pretrained: false` and expect much less from
   twelve epochs.
3. Choose the **GPU** accelerator.

**To continue an interrupted run**, attach the previous session's *output* as an input dataset
as well. Kaggle wipes `/kaggle/working` between sessions, so the checkpoint has to come back in
through the door it left by; the third cell copies the last one across before training starts.
Nothing else needs saying — the run reads the directory, sees which epochs are already done and
carries on from the next one.

In [ ]:
!ls /kaggle/input
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# torch and torchvision are already on the image, so the detector extra costs nothing here.
!git clone --depth 1 https://github.com/esamoun/dark-vessel-detection.git /kaggle/working/repo
!pip install -q -e "/kaggle/working/repo[detector]"

In [ ]:
# Bring back the last checkpoint of a previous session, if one is attached as an input.
# Kaggle's working directory does not survive a session; the run's resume does, provided the
# file is put back where the config looks for it.
import shutil
from pathlib import Path

working = Path("/kaggle/working/checkpoints")
attached = sorted(Path("/kaggle/input").glob("*/checkpoints/epoch-*.pt"))

if attached:
    working.mkdir(parents=True, exist_ok=True)
    shutil.copy2(attached[-1], working / attached[-1].name)
    shutil.copy2(attached[-1].parent.parent / "metrics.json", "/kaggle/working/metrics.json")
    print(f"resuming from {attached[-1].name}")
else:
    print("no checkpoint attached: this is the first session of the run")

In [ ]:
!darkvessel train --config /kaggle/working/repo/configs/train.yaml

When the session ends, **Save Version** so that `/kaggle/working` becomes an output dataset:
the checkpoints and `metrics.json` in it are what the next session resumes from, and
`metrics.json` is what the numbers in the README are copied out of. It is plain JSON and needs
neither torch nor a GPU to read.